In [ ]:
import os
from typing import TypedDict

from langchain_core.messages.tool import tool_call_chunk
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

In [ ]:
class AgentState(TypedDict):
    task: str
    status: str
    logs: list[str]

In [ ]:
def plan_node(state: AgentState):
    return {"status": "Planning", "logs": state["logs"] + ["Created a plan."]}

def execute_node(state: AgentState):
    return {"status": "Done", "logs": state["logs"] + ["Executed the plan."]}


In [ ]:
from IPython.display import display, Image

builder = StateGraph(AgentState)
builder.add_node("plan", plan_node)
builder.add_node("execute", execute_node)
builder.add_edge(START, "plan")
builder.add_edge("plan", "execute")
builder.add_edge("execute", END)

graph = builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))



In [ ]:
inputs = {"task": "Build an app", "logs": []}
for chunk in graph.stream(inputs,stream_mode="values"):
    print("---- State Snapshot ----")
    print(chunk)

In [ ]:
from langgraph.types import StreamWriter
from langgraph.config import get_stream_writer
import time
def slow_execute_node(state: AgentState):
    writer = get_stream_writer()
    writer({"progress": "10% Complete", "detail": "Logging servers..."})
    time.sleep(3)
    writer({"progress": "25% Complete", "detail": "Configuring servers..."})
    time.sleep(2)
    writer({"progress": "90% Complete", "detail": "Restarting servers..."})
    time.sleep(1)
    return {"status": "Done", "logs": state["logs"] + ["Infrastructure complete."]}


In [ ]:
from IPython.display import display, Image

builder = StateGraph(AgentState)
builder.add_node("plan", plan_node)
builder.add_node("slow_execute_node", slow_execute_node)
builder.add_edge(START, "plan")
builder.add_edge("plan", "slow_execute_node")
builder.add_edge("slow_execute_node", END)

graph = builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))



In [ ]:
for chunk in graph.stream(inputs, stream_mode=["custom", "updates"]):
    print("--- Custom Event Received ---")
    print(chunk)

In [ ]:
from typing import Annotated, TypedDict
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.config import get_stream_writer

# Define state that supports appending messages
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


llm = ChatOllama(
    model="llama3.2:latest",
    temperature=0,
    base_url="http://localhost:11434" # Default local Ollama port
)

def ai_agent_node(state: AgentState):

    writer = get_stream_writer()
    # Emit a custom status update
    writer({"status": "thinking", "message": "Llama 3 is processing your query..."})

    # Invoke the LLM
    response = llm.invoke(state["messages"])

    # Emit another custom update when the model finishes its output
    writer({"status": "complete", "message": "Llama 3 response finished."})

    return {"messages": [response]}

# Assemble the Graph
builder = StateGraph(AgentState)
builder.add_node("agent", ai_agent_node)
builder.add_edge(START, "agent")
builder.add_edge("agent", END)
graph = builder.compile()

# --- STREAM LOOP EXECUTOR ---
inputs = {"messages": [{"role": "user", "content": "Write a 1-sentence motivational quote."}]}

# Pass a list containing both modes
for mode, chunk in graph.stream(inputs, stream_mode=["messages", "custom"]):

    # Handle the custom progress pings
    if mode == "custom":
        print(f"\n⚡ [STATUS UPDATE]: {chunk['message']}")

    # Handle the raw Llama 3 text tokens
    elif mode == "messages":
        message, metadata = chunk

        # Only print tokens originating from the agent node
        if metadata.get("langgraph_node") == "agent" and message.content:
            print(message.content, end="", flush=True)
